In [ ]:
# Example 1: Basic Profile Analysis and Graph Generation
import cProfile
import pstats
import io
import tempfile
import os

import gprof2dot

print("Basic profile analysis and graph generation:")
print("=" * 43)

# Create sample functions to profile
def fibonacci(n):
    """Recursive fibonacci - inefficient for demonstration"""
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

def calculate_sum(n):
    """Simple summation function"""
    return sum(range(n))

def mixed_operations():
    """Function with multiple operations for profiling"""
    result = 0
    result += fibonacci(10)  # Recursive calls
    result += calculate_sum(1000)  # Loop operations
    result += len("profiling test") * 42  # String operations
    return result

print("1. Generating profile data:")
# Profile the mixed operations function
profiler = cProfile.Profile()
profiler.enable()

# Run the function multiple times to get meaningful data
for i in range(5):
    result = mixed_operations()

profiler.disable()
print(f"   Profiled function result: {result}")

# Save profile to string buffer
string_buffer = io.StringIO()
stats = pstats.Stats(profiler, stream=string_buffer)
stats.sort_stats('cumulative')
stats.print_stats(10)  # Top 10 functions

profile_output = string_buffer.getvalue()
print(f"   Profile lines generated: {len(profile_output.splitlines())}")
print(f"   Sample profile output:")
print(f"   {profile_output.splitlines()[2] if len(profile_output.splitlines()) > 2 else 'N/A'}")

# Summary of the collected profile
print(f"\n2. Profile data structure:")
print(f"   Total function calls: {stats.total_calls}")
print(f"   Primitive calls: {stats.prim_calls}")
print(f"   Total time: {stats.total_tt:.6f} seconds")

# Load the saved profile with gprof2dot's pstats parser
print(f"\n3. gprof2dot profile model:")
with tempfile.TemporaryDirectory() as tmp_dir:
    pstats_path = os.path.join(tmp_dir, "example1.pstats")
    profiler.dump_stats(pstats_path)
    profile = gprof2dot.PstatsParser(pstats_path).parse()

function_names = sorted(function.name for function in profile.functions.values())
print(f"   Functions parsed: {len(function_names)}")
for wanted in ("fibonacci", "calculate_sum", "mixed_operations"):
    matches = [name for name in function_names if name.endswith(":" + wanted)]
    print(f"   {wanted}: {matches}")
    assert matches, f"{wanted} missing from the gprof2dot profile: {function_names}"
fib = next(f for f in profile.functions.values() if f.name.endswith(":fibonacci"))
# fibonacci(10) makes 177 calls in total, and it ran 5 times
print(f"   fibonacci calls: {fib.called}")
assert fib.called == 5 * 177, fib.called

print("\nBasic profile analysis completed successfully!")

In [ ]:
# Example 2: Call Graph Analysis and DOT Format Generation
import cProfile
import pstats
import tempfile
import os
from io import StringIO

import gprof2dot

print("Call graph analysis and DOT format generation:")
print("=" * 46)

# Create a more complex example with nested function calls
class DataProcessor:
    def __init__(self, data_size):
        self.data_size = data_size
        self.processed_count = 0
    
    def process_data(self):
        """Main processing function"""
        data = self._generate_data()
        filtered = self._filter_data(data)
        return self._aggregate_data(filtered)
    
    def _generate_data(self):
        """Generate sample data"""
        return [i ** 2 for i in range(self.data_size)]
    
    def _filter_data(self, data):
        """Filter data based on criteria"""
        return [x for x in data if x % 3 == 0]
    
    def _aggregate_data(self, data):
        """Aggregate filtered data"""
        self.processed_count += 1
        return sum(data) / len(data) if data else 0

print("1. Creating complex call hierarchy:")
processor = DataProcessor(100)

# Profile the complex operations
profiler = cProfile.Profile()
profiler.enable()

results = []
for i in range(3):
    result = processor.process_data()
    results.append(result)

profiler.disable()
print(f"   Processing results: {results}")
print(f"   Processed count: {processor.processed_count}")

# Analyze the call graph structure
print(f"\n2. Call graph analysis:")
stats = pstats.Stats(profiler)
stats.sort_stats('cumulative')

# Get call graph information
total_calls = stats.total_calls
prim_calls = stats.prim_calls
print(f"   Total function calls: {total_calls}")
print(f"   Primitive calls: {prim_calls}")
print(f"   Recursive calls: {total_calls - prim_calls}")

# Analyze top functions by time
print(f"\n3. Top functions by cumulative time:")
string_io = StringIO()
stats.print_stats(5)
stats.stream = string_io
stats.print_stats(5)

# Write the call graph in DOT format with gprof2dot
print(f"\n4. DOT graph generated by gprof2dot:")
with tempfile.TemporaryDirectory() as tmp_dir:
    pstats_path = os.path.join(tmp_dir, "example2.pstats")
    profiler.dump_stats(pstats_path)
    profile = gprof2dot.PstatsParser(pstats_path).parse()

dot_buffer = StringIO()
gprof2dot.DotWriter(dot_buffer).graph(profile, gprof2dot.themes["color"])
dot_text = dot_buffer.getvalue()
dot_lines = dot_text.splitlines()
edge_lines = [line for line in dot_lines if " -> " in line]
print(f"   DOT lines: {len(dot_lines)}, edges: {len(edge_lines)}")
print(f"   First line: {dot_lines[0]}")
assert dot_text.startswith("digraph"), dot_text[:80]
for name in ("process_data", "_generate_data", "_filter_data", "_aggregate_data"):
    assert name in dot_text, f"{name} missing from the DOT output"
assert edge_lines, "no call edges in the DOT output"

print("\nCall graph analysis and DOT generation completed successfully!")

In [ ]:
# Example 3: Performance Hotspot Detection and Filtering
import cProfile
import pstats
import time
import random

print("Performance hotspot detection and filtering:")
print("=" * 42)

# Create functions with different performance characteristics
def fast_function():
    """Fast operation"""
    return sum(range(10))

def medium_function():
    """Medium speed operation"""
    return sum(i**2 for i in range(100))

def slow_function():
    """Slow operation with nested loops"""
    result = 0
    for i in range(50):
        for j in range(20):
            result += i * j
    return result

def very_slow_function():
    """Very slow operation"""
    time.sleep(0.01)  # Simulate I/O or slow computation
    return 42

def mixed_workload():
    """Function calling others with different frequencies"""
    results = []
    
    # Call fast function many times
    for _ in range(100):
        results.append(fast_function())
    
    # Call medium function moderate times
    for _ in range(20):
        results.append(medium_function())
    
    # Call slow function few times
    for _ in range(5):
        results.append(slow_function())
    
    # Call very slow function once
    results.append(very_slow_function())
    
    return sum(results)

print("1. Profiling workload with different performance characteristics:")
profiler = cProfile.Profile()
profiler.enable()

total_result = mixed_workload()

profiler.disable()
print(f"   Workload result: {total_result}")

# Analyze performance hotspots
print(f"\n2. Performance hotspot analysis:")
stats = pstats.Stats(profiler)

# Sort by different metrics to identify hotspots
metrics = ['cumulative', 'tottime', 'calls']
for metric in metrics:
    stats.sort_stats(metric)
    print(f"   Top 3 functions by {metric}:")
    
    # Get function statistics - note: stats.items() returns (key, (cc, nc, tt, ct, callers))
    func_stats = list(stats.stats.items())[:3]
    for (file, line, func), (cc, nc, tt, ct, callers) in func_stats:
        print(f"     {func}: calls={nc}, tottime={tt:.4f}s, cumtime={ct:.4f}s")

# Filter by cumulative time, as gprof2dot's --node-thres option does for graph nodes
print(f"\n3. Filtering for graph generation:")
threshold_time = 0.001  # 1ms threshold
significant_functions = []

for (file, line, func), (cc, nc, tt, ct, callers) in stats.stats.items():
    if ct > threshold_time:  # Filter by cumulative time
        significant_functions.append({
            'name': func,
            'calls': nc,
            'total_time': tt,
            'cumulative_time': ct,
            'per_call': ct / nc if nc > 0 else 0
        })

print(f"   Functions above {threshold_time}s threshold: {len(significant_functions)}")
for func_data in significant_functions[:5]:
    print(f"     {func_data['name']}: {func_data['cumulative_time']:.4f}s "
          f"({func_data['calls']} calls)")

print("\nPerformance hotspot detection completed successfully!")

In [ ]:
# Example 4: Multi-Format Profile Support and Graph Customization
import cProfile
import pstats
import tempfile
import io
import os
import json
from typing import Dict, List, Any

import gprof2dot

print("Multi-format profile support and graph customization:")
print("=" * 53)

# Create a comprehensive profiling scenario
class DatabaseSimulator:
    def __init__(self):
        self.connections = []
        self.query_cache = {}
    
    def connect(self, connection_id):
        """Simulate database connection"""
        self.connections.append(connection_id)
        return f"conn_{connection_id}"
    
    def execute_query(self, query, use_cache=True):
        """Simulate query execution with caching"""
        if use_cache and query in self.query_cache:
            return self.query_cache[query]
        
        # Simulate query processing time
        result = self._process_query(query)
        if use_cache:
            self.query_cache[query] = result
        return result
    
    def _process_query(self, query):
        """Internal query processing"""
        # Simulate different query complexities
        if "SELECT" in query:
            return self._select_operation(query)
        elif "INSERT" in query:
            return self._insert_operation(query)
        else:
            return self._complex_operation(query)
    
    def _select_operation(self, query):
        """Simulate SELECT operation"""
        return [{"id": i, "data": f"row_{i}"} for i in range(10)]
    
    def _insert_operation(self, query):
        """Simulate INSERT operation"""
        return {"inserted": 1, "id": len(self.connections)}
    
    def _complex_operation(self, query):
        """Simulate complex operation"""
        result = 0
        for i in range(100):
            result += i ** 2
        return {"computed": result}

print("1. Profiling database simulation scenario:")
db = DatabaseSimulator()

profiler = cProfile.Profile()
profiler.enable()

# Simulate various database operations
for conn_id in range(3):
    connection = db.connect(conn_id)
    
    # Execute different types of queries
    queries = [
        "SELECT * FROM users",
        "INSERT INTO users VALUES (1, 'Alice')",
        "SELECT * FROM users",  # This should use cache
        "UPDATE users SET name='Bob' WHERE id=1"
    ]
    
    for query in queries:
        result = db.execute_query(query)

profiler.disable()
print(f"   Database connections: {len(db.connections)}")
print(f"   Cache entries: {len(db.query_cache)}")

# Analyze and format profile data
print(f"\n2. Profile data analysis and formatting:")
stats = pstats.Stats(profiler)
stats.sort_stats('cumulative')

# Extract profile information for custom formatting
profile_data = {
    'summary': {
        'total_calls': stats.total_calls,
        'primitive_calls': stats.prim_calls,
        'total_time': stats.total_tt
    },
    'functions': []
}

for (filename, line_num, func_name), (cc, nc, tt, ct, callers) in list(stats.stats.items())[:10]:
    profile_data['functions'].append({
        'name': func_name,
        'filename': filename.split('/')[-1] if '/' in filename else filename,
        'line': line_num,
        'calls': nc,
        'total_time': tt,
        'cumulative_time': ct,
        'time_per_call': ct / nc if nc > 0 else 0
    })

print(f"   Profile summary:")
print(f"     Total calls: {profile_data['summary']['total_calls']}")
print(f"     Function entries: {len(profile_data['functions'])}")

# gprof2dot's input formats and colour themes
print(f"\n3. gprof2dot input formats and themes:")
print(f"   Input formats: {sorted(gprof2dot.formats)}")
print(f"   Themes: {sorted(gprof2dot.themes)}")
assert "pstats" in gprof2dot.formats, sorted(gprof2dot.formats)
assert {"color", "pink", "gray", "bw", "print"} <= set(gprof2dot.themes), sorted(gprof2dot.themes)

print(f"\n4. Rendering the profile with two themes:")
with tempfile.TemporaryDirectory() as tmp_dir:
    pstats_path = os.path.join(tmp_dir, "example4.pstats")
    profiler.dump_stats(pstats_path)
    profile = gprof2dot.PstatsParser(pstats_path).parse()

for theme_name in ("color", "bw"):
    dot_buffer = io.StringIO()
    gprof2dot.DotWriter(dot_buffer).graph(profile, gprof2dot.themes[theme_name])
    dot_text = dot_buffer.getvalue()
    print(f"   {theme_name}: {len(dot_text.splitlines())} DOT lines")
    assert dot_text.startswith("digraph"), dot_text[:80]
    assert "execute_query" in dot_text and "_select_operation" in dot_text, theme_name

print("\nMulti-format profile support completed successfully!")

In [ ]:
# Example 5: Production Performance Analysis Pipeline
import cProfile
import pstats
import tempfile
import os
import subprocess
import sys
import time
from contextlib import contextmanager

print("Production performance analysis pipeline:")
print("=" * 41)

# Production-ready profiling context manager
@contextmanager
def profile_context(output_file=None):
    """Context manager for clean profiling"""
    profiler = cProfile.Profile()
    profiler.enable()
    
    start_time = time.time()
    try:
        yield profiler
    finally:
        profiler.disable()
        end_time = time.time()
        
        if output_file:
            profiler.dump_stats(output_file)
        
        print(f"   Profiling session: {end_time - start_time:.4f}s")

# Production application simulator
class WebApplicationSimulator:
    def __init__(self):
        self.request_count = 0
        self.cache = {}
        self.db_calls = 0
    
    def handle_request(self, request_type, payload_size=100):
        """Handle different types of web requests"""
        self.request_count += 1
        
        if request_type == 'api':
            return self._handle_api_request(payload_size)
        elif request_type == 'static':
            return self._handle_static_request(payload_size)
        else:
            return self._handle_dynamic_request(payload_size)
    
    def _handle_api_request(self, payload_size):
        """Handle API request with database operations"""
        self.db_calls += 1
        data = self._simulate_database_query()
        response = self._serialize_response(data, payload_size)
        return response
    
    def _handle_static_request(self, payload_size):
        """Handle static file requests with caching"""
        cache_key = f"static_{payload_size}"
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        content = self._generate_static_content(payload_size)
        self.cache[cache_key] = content
        return content
    
    def _handle_dynamic_request(self, payload_size):
        """Handle dynamic content generation"""
        template_data = self._load_template_data()
        content = self._render_template(template_data, payload_size)
        return content
    
    def _simulate_database_query(self):
        """Simulate database query with processing time"""
        return [{"id": i, "value": i ** 2} for i in range(50)]
    
    def _serialize_response(self, data, size):
        """Serialize response data"""
        return {"data": data[:size], "count": len(data)}
    
    def _generate_static_content(self, size):
        """Generate static content"""
        return "x" * size
    
    def _load_template_data(self):
        """Load template data"""
        return {"title": "Dashboard", "items": list(range(20))}
    
    def _render_template(self, template_data, size):
        """Render template with data"""
        base = f"{template_data['title']}: {len(template_data['items'])} items"
        return base + ("." * size)

print("1. Production application profiling:")
app = WebApplicationSimulator()

# Profile realistic workload
with tempfile.NamedTemporaryFile(suffix='.prof', delete=False) as temp_file:
    temp_profile_file = temp_file.name

with profile_context(temp_profile_file) as profiler:
    # Simulate mixed workload
    request_types = ['api', 'static', 'dynamic', 'api', 'static']
    payload_sizes = [100, 50, 200, 75, 50]
    
    for req_type, size in zip(request_types, payload_sizes):
        response = app.handle_request(req_type, size)

print(f"   Requests handled: {app.request_count}")
print(f"   Database calls: {app.db_calls}")
print(f"   Cache entries: {len(app.cache)}")

# Production performance analysis
print(f"\n2. Performance analysis and reporting:")
stats = pstats.Stats(temp_profile_file)
stats.sort_stats('cumulative')

# Generate performance report
performance_report = {
    'execution_summary': {
        'total_calls': stats.total_calls,
        'primitive_calls': stats.prim_calls,
        'total_time': stats.total_tt
    },
    'top_functions': [],
    'bottlenecks': [],
    'recommendations': []
}

# Identify top functions and bottlenecks
for (file, line, func), (cc, nc, tt, ct, callers) in list(stats.stats.items())[:5]:
    func_data = {
        'function': func,
        'calls': nc,
        'total_time': tt,
        'cumulative_time': ct,
        'avg_time_per_call': ct / nc if nc > 0 else 0
    }
    performance_report['top_functions'].append(func_data)
    
    # Identify potential bottlenecks
    if ct > 0.001 and nc > 0:  # Functions taking > 1ms total
        if ct / nc > 0.0001:  # Average > 0.1ms per call
            performance_report['bottlenecks'].append({
                'function': func,
                'issue': 'High per-call execution time',
                'avg_time': ct / nc
            })

# Generate recommendations
if app.db_calls > 3:
    performance_report['recommendations'].append(
        "Consider database connection pooling or query optimization"
    )

if len(app.cache) < app.request_count:
    performance_report['recommendations'].append(
        "Implement more aggressive caching for static content"
    )

print(f"   Performance report generated:")
print(f"     Functions analyzed: {len(performance_report['top_functions'])}")
print(f"     Bottlenecks identified: {len(performance_report['bottlenecks'])}")
print(f"     Recommendations: {len(performance_report['recommendations'])}")

# Render the saved profile with the gprof2dot command line (-n/-e 0 keep every node and edge)
print(f"\n3. gprof2dot command line on the saved profile:")
cli = subprocess.run(
    [sys.executable, "-m", "gprof2dot", "-f", "pstats", "-n", "0", "-e", "0", temp_profile_file],
    capture_output=True, text=True, check=True,
)
print(f"   DOT output: {len(cli.stdout.splitlines())} lines")
assert cli.stdout.startswith("digraph"), cli.stdout[:80]
assert "handle_request" in cli.stdout and "_handle_api_request" in cli.stdout, "profiled functions missing from the DOT output"

# Clean up
os.unlink(temp_profile_file)

print(f"\n4. Production deployment considerations:")
print(f"   ✓ Profile data collection automated")
print(f"   ✓ Performance metrics extracted")
print(f"   ✓ Bottleneck detection implemented")
print(f"   ✓ Actionable recommendations generated")
print(f"   ✓ Integration-ready reporting pipeline")

print("\nProduction performance analysis pipeline completed successfully!")